# Visualization 1 (v3) — Pick the Number First, Then the Names

**Goal:** Show the evolution of French baby name popularity from 1900 to 2020.  
All names start as faint gray lines. The user **first chooses how many names to compare**, and exactly that many name dropdowns appear to fill in.

**Design (from 3_Visualizations.pdf)**
- X-axis: Year (1900–2020)
- Y-axis: Total births per year (summed across all departments and both genders)
- Background: all names as thin gray lines (opacity ≈ 15 %)
- Highlight: selected names drawn thick and colored, with end-of-line labels
- Interaction: a **"How many names?" selector** (1 … `MAX_SLOTS`) drives a control panel that renders that many name dropdowns on the fly. Each dropdown lists **every name** in the dataset (alphabetical, with type-ahead).

**Why a custom HTML page?** A compiled Vega-Lite chart always renders *all* of its bound inputs — a parameter cannot show or hide them. So this version exports the chart with unbound `name1 … nameN` parameters and wraps it in a small custom page whose JavaScript builds the dropdowns dynamically and pushes the chosen names into the chart via `view.signal(...)`.

**Data:** [dpt2020.csv](https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv)  
Columns: `sexe` (1=M, 2=F) · `preusuel` (name) · `annais` (year) · `dpt` (department code) · `nombre` (count)

In [ ]:
import pandas as pd
import altair as alt

print(f"pandas  {pd.__version__}")
print(f"altair  {alt.__version__}")

## 1 · Load & preprocess

In [ ]:
URL = "https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv"

df = pd.read_csv(URL, sep=';', dtype={'annais': str, 'dpt': str})
print(f"Raw rows: {len(df):,}")
df.head()

In [ ]:
import unicodedata

# Drop the aggregate 'rare names' catch-all bucket
df = df[df['preusuel'] != '_PRENOMS_RARES']

# Drop rows with unknown year ('XXXX') and cast to int
df = df[df['annais'] != 'XXXX'].copy()
df['annais'] = df['annais'].astype(int)
df = df[(df['annais'] >= 1900) & (df['annais'] <= 2020)]

# Normalize accents so spelling variants collapse into one name:
# LÉO → LEO, MAËL → MAEL, NOÉMIE → NOEMIE, etc. INSEE stores accented and
# unaccented forms as separate records, which otherwise splits a single name
# into two short, incomplete lines. Stripping diacritics merges them.
def strip_accents(s):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )

df['preusuel'] = df['preusuel'].map(strip_accents)

# Aggregate: total births per (name, year) — sums across all departments,
# genders, AND accent variants of the same name.
yearly = (
    df.groupby(['preusuel', 'annais'])['nombre']
    .sum()
    .reset_index()
    .rename(columns={'preusuel': 'name', 'annais': 'year', 'nombre': 'count'})
)

print(f"Unique names : {yearly['name'].nunique():,}")
print(f"Year range   : {yearly['year'].min()} – {yearly['year'].max()}")
print(f"Total rows   : {len(yearly):,}")

In [ ]:
# Keep only the top-N names by cumulative births to keep the chart responsive.
TOP_N = 300

top_names = (
    yearly.groupby('name')['count']
    .sum()
    .nlargest(TOP_N)
    .index.tolist()
)

# Exclude count <= 3: INSEE records rare occurrences as exactly 3 (privacy floor),
# which causes hundreds of names to pile up on the same Y position and form a
# visible horizontal strip at the bottom of the spaghetti chart.
df_plot = yearly[(yearly['name'].isin(top_names)) & (yearly['count'] > 3)].copy()
print(f"Plotting {TOP_N} names × {yearly['year'].nunique()} years = {len(df_plot):,} rows")

## 2 · Build the chart

In [ ]:
# Lift Altair's default 5 000-row cap
alt.data_transformers.disable_max_rows()

# ── Two separate sources ──────────────────────────────────────────────────────
# Both layers use the same count > 3 floor so no data point falls below domainMin=4.
yearly_fg = yearly[yearly['count'] > 3]
base_bg = alt.Chart(df_plot)    # top-300 gray backdrop  (already filtered count > 3)
base_fg = alt.Chart(yearly_fg)  # all names, highlighted (filtered count > 3)

# ── Parameters WITHOUT bindings ──────────────────────────────────────────────
# name1 … nameMAX_SLOTS exist as plain signals (no native <select>). The custom
# HTML page (next save cell) drives them via view.signal() from dropdowns it
# builds dynamically. MAX_SLOTS is the upper bound the user can dial up to.
all_names = sorted(yearly_fg['name'].unique().tolist())

MAX_SLOTS   = 10
DEFAULT_NUM = 3                       # dropdowns shown on first load
defaults    = ['LEO', 'ALICE', 'MAEL']  # pre-filled names for the first slots

slot_params = [
    alt.param(f'name{i}', value=(defaults[i - 1] if i - 1 < len(defaults) else ''))
    for i in range(1, MAX_SLOTS + 1)
]

# Toggle for the gray background spaghetti. The custom HTML page exposes this as
# a checkbox and drives it via view.signal('show_bg', ...). The bg layer is kept
# or dropped with transform_filter('show_bg'): when the signal is false the
# layer renders nothing, masking all the dark-gray backdrop curves at once.
show_bg = alt.param('show_bg', value=True)

# Highlight a line if its name matches ANY slot. Empty slots never match.
param_array = '[' + ', '.join(f'name{i}' for i in range(1, MAX_SLOTS + 1)) + ']'
SELECT_FILTER = f'indexof({param_array}, datum.name) >= 0'

x_enc = alt.X('year:Q', title='Year', axis=alt.Axis(format='d', tickCount=13))

# domainMin=4 matches the count > 3 filter applied to both layers, so the axis
# floor aligns with the lowest data point and no line can overflow below the X-axis.
y_enc = alt.Y(
    'count:Q',
    title='Total births (log scale)',
    scale=alt.Scale(type='log', zero=False, domainMin=4, nice=False),
    axis=alt.Axis(format='~s')
)

TOOLTIP = [
    alt.Tooltip('name:N',  title='Name'),
    alt.Tooltip('year:Q',  title='Year',         format='d'),
    alt.Tooltip('count:Q', title='Total births', format=',')
]

# ── Layer 1 · ALL top-300 names — faint gray spaghetti ───────────────────────
# transform_filter('show_bg') lets the checkbox in the HTML page hide every
# backdrop curve at once when the signal is toggled off.
bg = base_bg.mark_line(
    strokeWidth=0.8,
    opacity=0.15,
    color='#888888'
).encode(
    x=x_enc,
    y=y_enc,
    detail='name:N',
    tooltip=TOOLTIP
).transform_filter('show_bg')

# ── Layer 2 · SELECTED names — colored, thick lines ──────────────────────────
fg = (
    base_fg.mark_line(strokeWidth=2.8, opacity=0.9)
    .encode(
        x=x_enc,
        y=y_enc,
        color=alt.Color(
            'name:N',
            scale=alt.Scale(scheme='tableau20'),
            legend=alt.Legend(title='Selected names', orient='top-right')
        ),
        detail='name:N',
        tooltip=TOOLTIP
    )
    .transform_filter(SELECT_FILTER)
)

# ── Layer 3 · End-of-line labels ─────────────────────────────────────────────
lbl = (
    base_fg.mark_text(align='left', dx=6, dy=-3, fontSize=11, fontWeight='bold')
    .encode(
        x=x_enc,
        y=y_enc,
        color=alt.Color('name:N', scale=alt.Scale(scheme='tableau20'), legend=None),
        text='name:N'
    )
    .transform_filter(SELECT_FILTER)
    .transform_joinaggregate(max_year='max(year)', groupby=['name'])
    .transform_filter('datum.year == datum.max_year')
)

# ── Compose ───────────────────────────────────────────────────────────────────
chart = (
    (bg + fg + lbl)
    .add_params(*slot_params, show_bg)
    .properties(
        title=alt.TitleParams(
            text='French Baby Names — Evolution Over Time (1900–2020)',
            subtitle='Choose how many names to compare, then pick each from its dropdown — Y axis is logarithmic',
            anchor='start',
            fontSize=16,
            fontWeight='bold',
            subtitleFontSize=12,
            subtitleColor='#666666'
        ),
        width=760,
        height=450,
        padding={'left': 10, 'right': 90, 'top': 10, 'bottom': 10}
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridOpacity=0.2, gridColor='#e0e0e0')
)

chart

## 3 · Wrap in a custom HTML page with dynamic dropdowns

The cell below takes the compiled chart spec and embeds it in a small standalone
page. A **"How many names?"** selector controls how many name dropdowns are
rendered; each change pushes the chosen names into the chart through
`view.signal('name{i}', value)`.

In [ ]:
import json

spec = chart.to_dict()

# Custom standalone page: control panel (number selector + dynamic name
# dropdowns) wrapping the embedded Vega-Lite view. Placeholders are filled in
# with .replace() to avoid clashing with the braces in the CSS/JS.
TEMPLATE = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<title>French Baby Names — Choose How Many to Compare</title>
<script src="https://cdn.jsdelivr.net/npm/vega@6"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-lite@6.4.1"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-embed@7"></script>
<style>
  body { font-family: -apple-system, Segoe UI, Roboto, Helvetica, Arial, sans-serif; margin: 24px; color:#222; }
  #panel { background:#f7f7f9; border:1px solid #e2e2e8; border-radius:8px; padding:14px 16px; margin-bottom:16px; max-width:900px; }
  #panel .row { display:flex; align-items:center; gap:10px; margin-bottom:10px; }
  #panel label { font-weight:600; font-size:14px; }
  #name-controls { display:flex; flex-wrap:wrap; gap:10px 18px; }
  .ctrl { display:flex; align-items:center; gap:6px; }
  .ctrl label { font-weight:500; }
  select { font-size:14px; padding:3px 6px; }
  .hint { color:#666; font-size:12px; }
  .bg-toggle { display:flex; align-items:center; gap:6px; cursor:pointer; }
  .bg-toggle input { width:16px; height:16px; cursor:pointer; }
</style>
</head>
<body>
  <div id="panel">
    <div class="row">
      <label for="num-select">How many names to compare?</label>
      <select id="num-select"></select>
      <span class="hint">Tip: click a dropdown and start typing to jump to a name.</span>
    </div>
    <div class="row">
      <label class="bg-toggle" for="bg-toggle">
        <input type="checkbox" id="bg-toggle" checked/>
        Show background curves
      </label>
      <span class="hint">Uncheck to hide the faint gray spaghetti backdrop.</span>
    </div>
    <div id="name-controls"></div>
  </div>
  <div id="vis"></div>

<script type="text/javascript">
const SPEC = __SPEC__;
const NAMES = __NAMES__;
const MAX = __MAX__;
const DEFAULT_NUM = __DEFAULT_NUM__;
const DEFAULTS = __DEFAULTS__;

let selected = Array.from({length: MAX}, (_, i) => DEFAULTS[i] || '');
let view = null;

const optionsHTML = '<option value="">— none —</option>' +
  NAMES.map(n => '<option value="' + n + '">' + n + '</option>').join('');

function applyAllSignals(k) {
  for (let i = 1; i <= MAX; i++) {
    view.signal('name' + i, i <= k ? (selected[i-1] || '') : '');
  }
  view.runAsync();
}

function renderDropdowns(k) {
  const c = document.getElementById('name-controls');
  c.innerHTML = '';
  for (let i = 1; i <= k; i++) {
    const wrap = document.createElement('div');
    wrap.className = 'ctrl';
    const lab = document.createElement('label');
    lab.textContent = 'Name ' + i + ':';
    const sel = document.createElement('select');
    sel.innerHTML = optionsHTML;
    sel.value = selected[i-1] || '';
    sel.addEventListener('change', (e) => {
      selected[i-1] = e.target.value;
      view.signal('name' + i, e.target.value);
      view.runAsync();
    });
    wrap.appendChild(lab);
    wrap.appendChild(sel);
    c.appendChild(wrap);
  }
  applyAllSignals(k);
}

vegaEmbed('#vis', SPEC, {actions: false}).then((res) => {
  view = res.view;
  const numSel = document.getElementById('num-select');
  for (let n = 1; n <= MAX; n++) {
    const o = document.createElement('option');
    o.value = n; o.textContent = n;
    if (n === DEFAULT_NUM) o.selected = true;
    numSel.appendChild(o);
  }
  numSel.addEventListener('change', (e) => renderDropdowns(parseInt(e.target.value, 10)));

  // Show/hide the gray background spaghetti by driving the show_bg signal.
  const bgToggle = document.getElementById('bg-toggle');
  bgToggle.addEventListener('change', (e) => {
    view.signal('show_bg', e.target.checked);
    view.runAsync();
  });

  renderDropdowns(DEFAULT_NUM);
}).catch(console.error);
</script>
</body>
</html>'''

html = (TEMPLATE
        .replace('__SPEC__', json.dumps(spec))
        .replace('__NAMES__', json.dumps(all_names))
        .replace('__MAX__', str(MAX_SLOTS))
        .replace('__DEFAULT_NUM__', str(DEFAULT_NUM))
        .replace('__DEFAULTS__', json.dumps(defaults)))

with open('visualization_1_v3_spaghetti.html', 'w') as f:
    f.write(html)
print('Saved to visualization_1_v3_spaghetti.html')